# **YouTube Data Ingestion Pipeline**

This notebook accomplishes the following:
1. Pull official YouTube metadata via Google's YouTube Data API
2. Pull transcripts via an external API, provided a video ID

## Import Libraries and Environment Variables

In [19]:
import os
import re
import shutil
from pathlib import Path
from dotenv import load_dotenv
from googleapiclient.discovery import build
from typing import Union, List, Dict
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api.formatters import JSONFormatter

def clear_data_dirs():
    """Clear subdirectories: metadata, transcripts/cleaned, transcripts/raw."""
    dirs = [
        Path("data/metadata"),
        Path("data/transcripts/cleaned"),
        Path("data/transcripts/raw"),
    ]
    for d in dirs:
        if d.exists():
            shutil.rmtree(d)
        d.mkdir(parents=True, exist_ok=True)
    
# Initialize by cleaning data directories using function above...
clear_data_dirs()

In [20]:
# Load .env file
load_dotenv()
YT_API_KEY = os.getenv("YOUTUBE_DATA_API_KEY")

## Stage 1: Query Video IDs from YouTube Data API

In [21]:
"""
- Build a service object with appropriate credentials to pull data from YouTube, with the following constraints:
    - Ordering query results by view count (greatest to least)
    - Time frame limited to last 6 months
"""

youtube = build(serviceName="youtube", version="v3", developerKey=YT_API_KEY)

In [22]:
from datetime import datetime, timedelta, timezone
six_months_ago = (datetime.now(timezone.utc) - timedelta(days=180)).strftime("%Y-%m-%dT00:00:00Z")

# Returns a nested dictionary
response = youtube.search().list(
    q="personal health",
    part="snippet",
    type="video",
    maxResults=50,
    publishedAfter = six_months_ago,
    order="viewCount"
).execute()

### Exploratory Data Analysis (EDA)
- Previewing nested dictionary structure & relevant attributes for feature selection
    -  `response["items]`: list of search results (each formatted as a dict)





- Response attributes which aren't useful beyond validation:
    - `'kind', 'etag', 'nextPageToken', 'regionCode', 'pageInfo'`

In [23]:
# if it's a plain dict-like thing
print(f"Dictionary Structure:\n{response.keys()}")

from pprint import pprint
print("\nFull Response Dictionary Structure:\n")
pprint(response)

Dictionary Structure:
dict_keys(['kind', 'etag', 'nextPageToken', 'regionCode', 'pageInfo', 'items'])

Full Response Dictionary Structure:

{'etag': 'VfiYIxWDl_x3wSlDJgrn97hkyQs',
 'items': [{'etag': 'TDsa5I54kTcTy8WgQ9VRlSYr5k8',
            'id': {'kind': 'youtube#video', 'videoId': '0fb7SsSIlqM'},
            'kind': 'youtube#searchResult',
            'snippet': {'channelId': 'UCwAdQUuPT6laN-AQR17fe1g',
                        'channelTitle': 'Pranjal Kamra',
                        'description': 'Compare Best Health Plans with Ditto: '
                                       'https://ditto.sh/6zwj4x After the GST '
                                       'reforms, many of you asked us to bring '
                                       'back our annual ...',
                        'liveBroadcastContent': 'none',
                        'publishTime': '2025-10-28T13:24:43Z',
                        'publishedAt': '2025-10-28T13:24:43Z',
                        'thumbnails': {'default

In [24]:
# For each search result:
for item in response["items"]:
    # Print the video's title & channel title
    print(item["snippet"]["title"])
    print(item["snippet"]["channelTitle"])
    print("------------")

Best Health Insurance Plans 2026
Pranjal Kamra
------------
A Personal Health Statement from Peter Whittle, Founder &amp; Director of the New Culture Forum 
The New Culture Forum
------------
Dr. Tim Johnson details personal health battle
Good Morning America
------------
Event Recap：AI-Driven Preventive Healthcare Launch Event - Your Personal Health Butler Is Now Online
LIVE4WELL
------------
Doctor&#39;s Appointment - My Personal Health - Future Plans - Big Trip Coming
The Carpetbagger
------------
Shiori Shares a Personal Health Update【Hololive EN】
Novelitedits
------------
Harrison Smith Feeds the Local Media Jackals, Talks Personal Health Issue
Purple FTW! Podcast
------------
lBeyond Business: Ratan Tata&#39;s Personal Health Battle &amp; His Support System&quot; #shorts #trending #news
MH news
------------
Top 10 Health Benefits of Cardamom | এলাচ খাওয়ার উপকারিতা | Weight Loss &amp; Heart Health
Dr. Tasnim Health Care
------------
The Future of Personal Health Tech!
iJustine
--

In [25]:
# Verify # of items matches input response
if len(response["items"]) == 50:
    print("Number of results matches input search criterion!")
else:
    print(f"MISMATCH! Received only {len(response["items"])} video results.")

Number of results matches input search criterion!


## Stage 2: Transcript Extraction

Video IDs are extracted from the search response, passed to the transcript API, and saved per-video to `data/transcripts/raw/{video_id}.json` and `data/transcripts/cleaned/{video_id}.txt`.

### Fetch Video IDs from Search Response

In [26]:
query_video_ids = []

# For each search result
for item in response["items"]:
    query_video_ids.append(item['id']['videoId'])

# Print for validation
print(f"TOTAL # of Video IDs: {len(query_video_ids)}")

print("Sample First 5 Video IDs:\n-----------")
for i in range(0, 5):
    print(query_video_ids[i])

TOTAL # of Video IDs: 50
Sample First 5 Video IDs:
-----------
0fb7SsSIlqM
MZ65Z8papwg
9Zz6Nj0xwXM
kRLWY-u52jU
pP0cCa7CbQo


### Filtering for High-Impact Videos
For each video in queried set of IDs, check the video atttributes to filter only high-impact videos. 
- We set the criterion here so we need to justify that. 
- Return a subset of original queried IDs which fulfil the criterion. 

In [27]:

def chunk_video_ids(lst, n):
    """Breaks down any input series of video_ids into smaller subset of lists. Going to be useful to handle API request limits. """
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

    
def fetch_video_metadata(youtube, video_ids):
    """Helper function to get hydrated set of video metrics beyond search"""
    all_items = []

    for job in chunk_video_ids(video_ids, 50):
        resp = youtube.videos().list(
            part="snippet,statistics,contentDetails",
            id=",".join(job),
            maxResults = 50
        ).execute()

        all_items.extend(resp.get("items", []))
    
    return all_items

In [28]:
video_mds = fetch_video_metadata(youtube, query_video_ids)
print(len(video_mds), video_mds[0].keys())

50 dict_keys(['kind', 'etag', 'id', 'snippet', 'contentDetails', 'statistics'])


In [29]:
print("ContentDetails:")
pprint(video_mds[0]['contentDetails'])

print("\nStatistics")
pprint(video_mds[0]['statistics'])

print("\nSnippet")
pprint(video_mds[0]['snippet'])

ContentDetails:
{'caption': 'false',
 'contentRating': {},
 'definition': 'hd',
 'dimension': '2d',
 'duration': 'PT18M13S',
 'licensedContent': True,
 'projection': 'rectangular'}

Statistics
{'commentCount': '950',
 'favoriteCount': '0',
 'likeCount': '13665',
 'viewCount': '358394'}

Snippet
{'categoryId': '27',
 'channelId': 'UCwAdQUuPT6laN-AQR17fe1g',
 'channelTitle': 'Pranjal Kamra',
 'defaultAudioLanguage': 'hi',
 'defaultLanguage': 'en-IN',
 'description': 'Compare Best Health Plans with Ditto: '
                'https://ditto.sh/6zwj4x \n'
                '\n'
                'After the GST reforms, many of you asked us to bring back our '
                'annual insurance recommendation video early, so we’re back, '
                'starting with the Best Health Insurance Plans for 2026. \n'
                '\n'
                'In this video, we reveal the Best Health Insurance Plans for '
                '2026, shortlisted through a transparent and data-backed '
           

In [30]:
print(f"Snippet Keys: {video_mds[0]['snippet'].keys()}")

Snippet Keys: dict_keys(['publishedAt', 'channelId', 'title', 'description', 'thumbnails', 'channelTitle', 'tags', 'categoryId', 'liveBroadcastContent', 'defaultLanguage', 'localized', 'defaultAudioLanguage'])


- Factors from Videos.list() to add to video DB schema:
    - Snippet: `categoryId`
    - Statistics:  `commentCount, likeCount, viewCount`
    - ContentDetails: `duration`

In [31]:
### Compute Engagement Metrics for Filtration of Records
from datetime import datetime, timezone
import math

def compute_impact_features(vid_metadata):
    """Provided a video dictionary from videos().list() --> we want to return a dict with derived impact features."""

    stats = vid_metadata.get("statistics", {})
    snippet = vid_metadata.get("snippet", {})

    # Raw Engagement Data
    view_count = int(stats.get("viewCount", 0))
    comment_count = int(stats.get("commentCount", 0))
    like_count = int(stats.get("likeCount", 0))
    published_at = datetime.fromisoformat(
        snippet["publishedAt"].replace("Z", "+00:00")
    )
    days_since = max(
        (datetime.now(timezone.utc) - published_at).days,
        1  # avoid divide-by-zero
    )

    # Derived metrics from API provided data
    views_per_day = view_count / days_since
    comments_per_1kviews = ((comment_count / view_count) * 1000) if view_count > 0 else 0
    likes_per_1kviews = ((like_count / view_count) * 1000) if view_count > 0 else 0

    # Metric weightage (impact score calculation)
    reach = math.log10(view_count + 1)
    momentum = math.log10(views_per_day + 1)

    # Normalized engagement scores
    engagement = math.log10(comments_per_1kviews + 1) + math.log10(likes_per_1kviews + 1)
    impact_score = 0.45 * reach + 0.35 * momentum + 0.20 * engagement
    
    return {
        "video_id": vid_metadata["id"],
        "view_count": view_count,
        "views_per_day": views_per_day,
        "comments_per_1kviews": comments_per_1kviews,
        "likes_per_1kviews": likes_per_1kviews,
        "impact_score": impact_score,
    }

In [32]:
# Percentile-Based Filtering --> accomodates for cases when number of videos per call vary, so the cutoff is malleable with respect to video ids pulled.
def filter_ws_by_percentile(scored_videos, percentile=0.9):
    """
    Keeps videos above the given percentile of impact_score.
    Example: percentile=0.9 keeps top 10%.
    """

    if not scored_videos:
        return []

    # Sort by impact_score descending
    scored_videos.sort(
        key=lambda x: x["impact_score"],
        reverse=True
    )

    cutoff_index = int(len(scored_videos) * percentile)

    # Prevent empty return
    cutoff_index = min(cutoff_index, len(scored_videos) - 1)

    threshold_score = scored_videos[cutoff_index]["impact_score"]

    filtered = [
        v for v in scored_videos
        if v["impact_score"] >= threshold_score
    ]

    return filtered

In [33]:
MIN_COMMENTS_PER_1K = 2.5      # 0.25%
MIN_LIKES_PER_1K = 25         # 2.5%
MIN_VIEWS = 1000             

# Calculated each video's eval metrics
video_impact_metrics = [
    compute_impact_features(v)
    for v in video_mds
]

# Skip low-engagement / low-reach videos
eligible_videos = [
    v for v in video_impact_metrics
    if (
        v["view_count"] >= MIN_VIEWS and
        v["comments_per_1kviews"] >= MIN_COMMENTS_PER_1K and
        v["likes_per_1kviews"] >= MIN_LIKES_PER_1K
    )
]

# Filter for top 10%
high_impact_videos = filter_ws_by_percentile(
    eligible_videos,
    percentile=0.25
)

# Return final set of eligible video IDs
filtered_video_ids = [
    v["video_id"] for v in high_impact_videos
]

### Extract and Clean Transcripts
- Use external YT Transcripts API for 2-stage data ingestion
- Defined a helper function to collapse JSON structure and remove noise; resulting cleaned transcript is stored as a .txt object. 


In [34]:
# clean_transcript defined in imports cell above
def clean_transcript(transcript_data: Union[List[Dict], str]) -> str:
    """
    Clean a YouTube transcript by removing noise and formatting artifacts into .txt blob.
    """
    if isinstance(transcript_data, str):
        text = transcript_data
    else:
        text = " ".join([item.text for item in transcript_data])
    text = re.sub(r'^>\s*', '', text, flags=re.MULTILINE)
    text = re.sub(r'(?:^|\s)(?:[A-Z][a-z]*(?:\s+[A-Z][a-z]*)?)\s*:\s*', ' ', text)
    text = re.sub(r'\[[^\]]*\]', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\([^)]*\)', '', text)
    for pattern in [r'\b(?:um|uh|ugh|hmm)\b', r'\byou\s+know\b', r'\blike\b(?=\s+(?:he|she|they|it|the|a|i))']:
        text = re.sub(pattern, '', text, flags=re.IGNORECASE)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\s+([.,!?;:])', r'\1', text)
    text = re.sub(r'([.!?])\s+(?=[a-z])', lambda m: m.group(1) + ' ', text)
    text = text.strip()
    if text and text[0].islower():
        text = text[0].upper() + text[1:]
    return text

In [35]:
ytt_api = YouTubeTranscriptApi()
formatter = JSONFormatter()

Path("data/transcripts/raw").mkdir(parents=True, exist_ok=True)
Path("data/transcripts/cleaned").mkdir(parents=True, exist_ok=True)

print(f"Number of candidate video IDs to evaluate: {len(filtered_video_ids)}")

for video_id in filtered_video_ids:
    try:
        raw_transcript = ytt_api.fetch(video_id)
        json_formatted = formatter.format_transcript(raw_transcript, indent=2)
        Path(f"data/transcripts/raw/{video_id}.json").write_text(json_formatted, encoding="utf-8")
        cleaned = clean_transcript(raw_transcript)
        Path(f"data/transcripts/cleaned/{video_id}.txt").write_text(cleaned, encoding="utf-8")
    except Exception:
        print(f"No transcript available for Video ID: {video_id}")

Number of candidate video IDs to evaluate: 5
No transcript available for Video ID: 0fb7SsSIlqM


In [36]:
# Validate data that is being stored in the directories for now; make sure we have the same # of entries in both subidrectories (raw - cleaned)
raw_count = len([p for p in Path("data/transcripts/raw").iterdir() if p.is_file()])
cleaned_count = len([p for p in Path("data/transcripts/cleaned").iterdir() if p.is_file()])

if raw_count == cleaned_count:
    print(f"Verification successful: {raw_count} files in both directories.")
else:
    print(f"Mismatch: {raw_count} raw files, {cleaned_count} cleaned files.")

Verification successful: 4 files in both directories.


## Next Steps

See [README.md](README.md) for setup instructions and the full project workflow. The pipeline is headed in the following direction:

1. **Search enhancements** — Add views/impressions filtering via `videos.list` for more targeted discovery
2. **Batch processing** — Extend transcript extraction to all video IDs (currently processes the first video only)
3. **Supabase migration** — Move ingested data from local `data/` storage to Supabase (as noted in Data Storage above)
4. **Downstream use** — Cleaned transcripts are ready for LLM-based insight generation and analysis

Run this notebook after configuring `YOUTUBE_DATA_API_KEY` in `.env` (see README).